# M4-L1 — Nucleation–elongation cascade (rate equations)

**BruSafe / CAPSID · iGEM 2026 · Module 4 — Assembly**

This notebook is self-contained: the first code cell writes the simulation source to disk, so nothing has to be downloaded. It runs on Colab, in Jupyter, or on a laptop with `numpy`, `scipy` and `matplotlib`.

The theory and the bench protocols (§11) are described in [`M4_theory_compendium.pdf`](../../docs/M4_theory_compendium.pdf) in `capsid/docs/`.

**What this layer does.** Expression (Module 3) sets the total coat-protein concentration; the monomer–dimer equilibrium converts it into free dimer; and a 90-step master equation grows dimers into closed T=3 shells. Every reverse rate is derived from the forward rate by detailed balance, so the model cannot produce a yield optimum by tuning rates independently. The peaked yield curve is a prediction and not a fit.

**Runtime:** ~1 min for the full concentration sweep.

In [ ]:
# Colab / fresh machine: uncomment
# !pip -q install numpy scipy matplotlib
import numpy as np, matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

In [ ]:
%%writefile brusafe_synthesis.py
"""
BruSafe - MS2 VLP synthesis: expanded mathematical model
=========================================================
Two modules, self-contained (numpy + scipy only):

  Module 3+  Coat-protein SYNTHESIS: transcription / translation ODEs with an
             EXPLICIT reversible monomer<->dimer equilibrium.  The proposal's
             assumption D = P/2 is recovered as the tight-binding limit (Ni 1995).

  Module 4+  Capsid SELF-ASSEMBLY as an EXPLICIT sequential nucleation-elongation
             cascade over all 90 intermediate sizes (Zlotnick 1994; Endres &
             Zlotnick 2002; Ceres & Zlotnick 2002; Hagan 2010).  Reverse rates
             come from subunit-subunit contact free energies via detailed
             balance, so the model reproduces the nucleation lag and the
             concentration-dependent KINETIC TRAP that the lumped 3-state ODE
             in the main proposal cannot show.

All parameters carry literature-backed values; every rate is estimable without
experimental data (Mode A) and refinable against EMSA/SEC/light-scattering time
courses if available (Mode B).
"""
import numpy as np
from scipy.integrate import solve_ivp, odeint

RT    = 0.593    # kcal/mol at ~298 K
N_CAP = 90       # coat-protein dimers per T=3 MS2 capsid (Ni 1995)

# =====================================================================
# MODULE 3+  Expression with explicit reversible dimerisation
# =====================================================================
def induction(t, t_on=30.0, ramp=10.0):
    """Smooth inducer switch (IPTG / arabinose): 0 -> 1 around t_on."""
    return 1.0 / (1.0 + np.exp(-(t - t_on) / ramp))

def expression_ode(y, t, alpha, delta_m, mu, k_tl, delta_P, t_on):
    """dm/dt = alpha*u(t) - (delta_m+mu) m ;  dP/dt = 1e-3 k_tl m - (delta_P+mu) P
    m in nM, P in uM monomer-equivalents (1e-3 converts nM mRNA -> uM protein)."""
    m, P = y
    u = induction(t, t_on)
    dm = alpha * u - (delta_m + mu) * m
    dP = 1e-3 * k_tl * m - (delta_P + mu) * P
    return [dm, dP]

def simulate_expression(p, t_grid, t_on=30.0):
    delta_m = np.log(2) / p["t_half_mrna_min"]
    mu      = np.log(2) / p["t_double_min"]
    sol = odeint(expression_ode, [0.0, 0.0], t_grid,
                 args=(p["alpha_tx_nM_min"], delta_m, mu,
                       p["k_tl_permin"], p["delta_prot_permin"], t_on),
                 rtol=1e-8, atol=1e-10)
    m, P = sol[:, 0], sol[:, 1]
    return m, P

def expression_steady_state(p):
    """Closed form (u->1): m_ss = alpha/(delta_m+mu);
       P_ss = 1e-3 k_tl m_ss / (delta_P+mu).  Pen-and-paper estimable."""
    delta_m = np.log(2) / p["t_half_mrna_min"]
    mu      = np.log(2) / p["t_double_min"]
    m_ss = p["alpha_tx_nM_min"] / (delta_m + mu)
    P_ss = 1e-3 * p["k_tl_permin"] * m_ss / (p["delta_prot_permin"] + mu)
    return m_ss, P_ss

def monomer_dimer_split(P_tot_uM, Kd_dim_uM):
    """Equilibrium partition of total protein into free monomer M and dimer D:
         2D + M = P_tot ,  D = M^2 / Kd_dim.
    Solve (2/Kd)M^2 + M - P_tot = 0.  As Kd_dim -> 0 (tight dimer, Ni 1995),
    D -> P_tot/2, which is the assumption used in the main proposal."""
    a = 2.0 / Kd_dim_uM; b = 1.0; c = -P_tot_uM
    M = (-b + np.sqrt(b*b - 4*a*c)) / (2*a)
    D = M*M / Kd_dim_uM
    return M, D

# =====================================================================
# MODULE 4+  Explicit sequential nucleation-elongation cascade
#   x[j-1] = concentration (uM) of the j-mer, j = 1..N_CAP
#   x_1 is the free coat-protein DIMER (the assembly subunit)
#   reaction j:  X_j + X_1 <-> X_{j+1}   (j=1 uses x_1^2)
# =====================================================================
def contacts_for_step(j_target, n_c):
    """Subunit-subunit contacts formed when a subunit adds to make the j_target-mer.
    Pre-nucleus (<= n_c): 1 contact -> net UP-hill with the entropic penalty ->
      nucleation is rare (the free-energy barrier).
    Growth: 2 contacts.  Closing vertices (last ~8): 3 contacts -> strong drive
      to completion (Endres & Zlotnick 2002)."""
    if j_target <= n_c:            return 1
    if j_target >= N_CAP - 8:      return 3
    return 2

def build_rates(k_a, g_assoc, g_contact, n_c):
    """Forward on-rate k_a (uM^-1 min^-1); reverse from detailed balance.
      dG_add(j) = g_assoc + contacts(j)*g_contact
      g_assoc  > 0 : per-subunit association (translational/rotational) penalty
      g_contact< 0 : free energy per subunit-subunit contact (Ceres 2002)
      Kd(j) = exp(dG_add/RT) ;  b_{j+1} = k_a * Kd(j)."""
    f = np.full(N_CAP, k_a)
    b = np.zeros(N_CAP + 1)
    for j in range(1, N_CAP):
        c = contacts_for_step(j + 1, n_c)
        dG_add = g_assoc + c * g_contact
        b[j + 1] = k_a * np.exp(dG_add / RT)
    return f, b

def cascade_ode(t, x, f, b):
    x = np.clip(x, 0.0, None)
    x1 = x[0]
    J = np.zeros(N_CAP - 1)                # J[j-1] = net flux of reaction j
    J[0] = f[0] * x1 * x1 - b[2] * x[1]    # 2 X1 -> X2
    for j in range(2, N_CAP):
        J[j - 1] = f[j - 1] * x[j - 1] * x1 - b[j + 1] * x[j]
    dx = np.zeros(N_CAP)
    dx[0] = -2 * J[0] - np.sum(J[1:])      # subunit consumed by every step
    dx[1] = J[0] - (J[1] if N_CAP > 2 else 0.0)
    for k in range(3, N_CAP):
        dx[k - 1] = J[k - 2] - J[k - 1]
    dx[N_CAP - 1] = J[N_CAP - 2]           # complete capsid
    return dx

def simulate_cascade(D0_uM, k_a, g_assoc, g_contact, n_c, t_max=240.0, nt=600):
    """Returns t, capsid_frac(t), free_frac(t), inter_frac(t), full state X.
    Mass fractions are of the total subunit pool and sum to 1 at all times."""
    f, b = build_rates(k_a, g_assoc, g_contact, n_c)
    x0 = np.zeros(N_CAP); x0[0] = D0_uM
    t_eval = np.linspace(0, t_max, nt)
    sol = solve_ivp(cascade_ode, (0, t_max), x0, t_eval=t_eval,
                    args=(f, b), method="LSODA", rtol=1e-7, atol=1e-10)
    x = sol.y
    capsid_frac = N_CAP * x[-1] / D0_uM
    free_frac   = x[0] / D0_uM
    j = np.arange(1, N_CAP + 1)[:, None]
    inter_frac  = (np.sum(j * x, axis=0) - x[0] - N_CAP * x[-1]) / D0_uM
    return t_eval, capsid_frac, free_frac, inter_frac, x

# =====================================================================
# Literature-mean parameters (Mode A defaults)
# =====================================================================
PARAMS_SYNTH = {
    # Module 3 expression (Milo 2010; Bernstein 2002; Selinger 2003)
    "t_half_mrna_min":  (5.0,  (2.0, 8.0),  "Bernstein2002; Selinger2003"),
    "t_double_min":     (40.0, (25., 60.),  "Milo2010"),
    "alpha_tx_nM_min":  (6.0,  (1.0, 30.),  "Milo2010"),
    "k_tl_permin":      (4.0,  (1.0, 12.),  "Milo2010"),
    "delta_prot_permin":(0.002,(0.0, 0.01), "Milo2010"),
    "Kd_dim_uM":        (0.01, (1e-3, 1.0), "Ni1995 (tight obligate dimer)"),
    # Module 4 assembly (Zlotnick1994; Endres2002; Ceres2002; Hagan2010)
    "k_a_uM_min":       (3.0,  (0.4, 4.0),  "Ceres2002; Endres2002 (assoc on-rate)"),
    "g_assoc_kcal":     (6.0,  (4.0, 8.0),  "Zlotnick1994 (assoc entropy penalty)"),
    "g_contact_kcal":   (-4.0, (-5., -3.),  "Ceres2002 (per-contact free energy)"),
    "n_c_nucleus":      (4,    (3, 6),      "Endres2002 (nucleus size)"),
}

if __name__ == "__main__":
    p = {k: v[0] for k, v in PARAMS_SYNTH.items()}
    m_ss, P_ss = expression_steady_state(p)
    M, D = monomer_dimer_split(P_ss, p["Kd_dim_uM"])
    print(f"M3+ steady state: mRNA={m_ss:.1f} nM, protein={P_ss:.2f} uM, "
          f"dimer={D:.2f} uM ({2*D/P_ss:.3f} of mass; P/2={P_ss/2:.2f})")
    for D0 in (2.0, 10.0):
        t, cf, ff, itf, _ = simulate_cascade(
            D0, p["k_a_uM_min"], p["g_assoc_kcal"], p["g_contact_kcal"], p["n_c_nucleus"])
        print(f"M4+ D0={D0:5} uM: capsid@240={cf[-1]:.2f}  intermediate@240={itf[-1]:.2f}"
              f"  ({'trapped' if itf[-1] > cf[-1] else 'productive'})")


## 1. Steady-state expression and the monomer–dimer split

In [ ]:
import brusafe_synthesis as L1

p = {k: v[0] for k, v in L1.PARAMS_SYNTH.items()}
m_ss, P_ss = L1.expression_steady_state(p)
M, D = L1.monomer_dimer_split(P_ss, p['Kd_dim_uM'])
print(f'mRNA steady state   = {m_ss:.1f} nM')
print(f'total coat protein  = {P_ss:.2f} uM')
print(f'free dimer [D]      = {D:.2f} uM  ({2*D/P_ss:.1%} of protein mass)')
print(f'free monomer        = {M:.3f} uM')

## 2. One assembly trajectory

`simulate_cascade` returns the time grid and the full 90-species state. Species index 0 is free dimer, index 89 is the closed capsid.

In [ ]:
t, cap_t, free_t, inter_t, y = L1.simulate_cascade(D0_uM=2.0, k_a=p['k_a_uM_min'], g_assoc=p['g_assoc_kcal'],
                           g_contact=p['g_contact_kcal'], n_c=int(p['n_c_nucleus']))
j = np.arange(1, y.shape[0]+1)
mass = y * j[:, None]
frac = mass / mass.sum(axis=0)

fig, ax = plt.subplots(figsize=(6.2, 4))
ax.plot(t, frac[-1], lw=2, color='#0b6e5a', label='closed capsid')
ax.plot(t, frac[0],  lw=2, color='#c8863c', label='free dimer')
ax.plot(t, frac[1:-1].sum(axis=0), lw=2, color='#8a4f7d', label='partial shells')
ax.set_xlabel('time (min)'); ax.set_ylabel('fraction of coat mass')
ax.set_title(r'Assembly at $[D]_0$ = 2.0 $\mu$M'); ax.legend(frameon=False)
ax.set_ylim(0, 1); fig.tight_layout()
print('capsid mass fraction at 240 min:', round(float(frac[-1, -1]), 3))

## 3. The primary prediction: yield is peaked, not saturating

This is the curve that Experiment 1 measures. The grid points shown in bold in the protocol (2.25, 2.5, 2.75 and 3.0 µM) bracket the optimum, which is where the design carries the most information.

In [ ]:
dgrid = np.array([0.5, 1.0, 1.5, 2.0, 2.25, 2.5, 2.75, 3.0, 3.5, 4.0, 5.0, 7.0, 10.0])

rows = []
for D0 in dgrid:
    _, _, _, _, yy = L1.simulate_cascade(D0_uM=D0, k_a=p['k_a_uM_min'], g_assoc=p['g_assoc_kcal'],
                                         g_contact=p['g_contact_kcal'], n_c=int(p['n_c_nucleus']))
    x = yy[:, -1]; m = x * j; m = m / m.sum()
    part = m[1:-1]
    mean_sz = float((part * j[1:-1]).sum() / part.sum()) if part.sum() > 0 else np.nan
    rows.append((D0, m[-1], m[0], part.sum(), mean_sz))

import csv
with open('M4_L1_yield_curve.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['D0_uM','capsid_frac','free_dimer_frac',
                                   'partial_frac','mean_partial_size_dimers'])
    w.writerows([[f'{v:.4g}' for v in r] for r in rows])

for r in rows:
    print(f'{r[0]:6.2f} uM   capsid {r[1]*100:5.1f}%   free {r[2]*100:5.1f}%   '
          f'partial {r[3]*100:5.1f}%   mean partial {r[4]:.0f} dimers')

In [ ]:
D0s = np.array([r[0] for r in rows]); cap = np.array([r[1] for r in rows])
fre = np.array([r[2] for r in rows]); par = np.array([r[3] for r in rows])
k = int(np.argmax(cap))

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(D0s, cap*100, 'o-', lw=2, ms=5, color='#0b6e5a', label='closed capsid')
ax.plot(D0s, par*100, 's-', lw=1.6, ms=4, color='#8a4f7d', label='partial shells')
ax.plot(D0s, fre*100, '^-', lw=1.6, ms=4, color='#c8863c', label='free dimer')
ax.axvline(D0s[k], ls=':', color='0.5')
ax.annotate(f'optimum {D0s[k]:.2f} $\\mu$M\n{cap[k]*100:.0f}% capsid',
            xy=(D0s[k], cap[k]*100), xytext=(D0s[k]*1.5, 78),
            arrowprops=dict(arrowstyle='->', color='0.4'), fontsize=9)
ax.set_xscale('log'); ax.set_xlabel(r'initial dimer $[D]_0$ ($\mu$M)')
ax.set_ylabel('% of coat mass at 240 min')
ax.set_title('M4-L1 prediction: yield is peaked in concentration')
ax.legend(frameon=False, fontsize=9); ax.set_ylim(-3, 103)
fig.tight_layout(); fig.savefig('M4_L1_yield_curve.png', dpi=200)
print('optimum at', round(float(D0s[k]),2), 'uM; peak capsid fraction', round(float(cap[k]),3))

## 4. Identifiability: which parameter moves what

This cell turns the sweep from an optimisation into a measurement. `g_contact` moves the position of the optimum, `k_a` moves its height, and `n_c` is confounded with `g_contact`, so it must be fixed (or measured separately, for example from single-particle waiting times).

In [ ]:
def opt_for(gc=None, ka=None, nc=None):
    q = dict(k_a=ka or p['k_a_uM_min'], g_assoc=p['g_assoc_kcal'],
             g_contact=gc if gc is not None else p['g_contact_kcal'],
             n_c=int(nc or p['n_c_nucleus']))
    g = np.geomspace(0.2, 12, 22); ys = []
    for D0 in g:
        _, _, _, _, yy = L1.simulate_cascade(D0_uM=D0, **q)
        m = yy[:, -1] * j; ys.append(m[-1] / m.sum())
    ys = np.array(ys); i = int(np.argmax(ys))
    return float(g[i]), float(ys[i])

print('param            value    optimal [D]0    yield at optimum')
for gc in [-3.0, -3.5, -4.0, -4.5, -5.0]:
    d, yv = opt_for(gc=gc);  print(f'g_contact      {gc:6.1f}      {d:6.2f} uM        {yv:.2f}')
for ka in [0.4, 1.0, 3.0, 4.0]:
    d, yv = opt_for(ka=ka);  print(f'k_a            {ka:6.1f}      {d:6.2f} uM        {yv:.2f}')
for nc in [3, 4, 5, 6]:
    d, yv = opt_for(nc=nc);  print(f'n_c            {nc:6d}      {d:6.2f} uM        {yv:.2f}')

---
**Outputs written:** `M4_L1_yield_curve.csv`, `M4_L1_yield_curve.png`.

The CSV can be loaded into `M4_FIT_to_bench_data.ipynb` together with the measured points.